# Notebook 02 — Sensor-Only Baselines
**Goal:** Establish how well the 7 chemical/physical sensor features alone can classify litchi quality.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

SENSOR_COLS = ['Moisture', 'Firmness', 'pH', 'TSS', 'Ethanol', 'CO2', 'Formaldehyde']
TARGET_NAMES = ['Fresh', 'Formalin-Mixed', 'Rotten']
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Load Processed Splits

In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

X_train = train_df[SENSOR_COLS].values
y_train = train_df['label'].values
X_val = val_df[SENSOR_COLS].values
y_val = val_df['label'].values
X_test = test_df[SENSOR_COLS].values
y_test = test_df['label'].values

# Scale sensor features (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 2099, Val: 451, Test: 450


## 2. Random Forest

In [3]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='macro')
print(f"Random Forest  ->  Accuracy: {rf_acc*100:.2f}%  |  Macro F1: {rf_f1:.4f}")
print(classification_report(y_test, rf_pred, target_names=TARGET_NAMES))

Random Forest  ->  Accuracy: 100.00%  |  Macro F1: 1.0000
                precision    recall  f1-score   support

         Fresh       1.00      1.00      1.00       150
Formalin-Mixed       1.00      1.00      1.00       150
        Rotten       1.00      1.00      1.00       150

      accuracy                           1.00       450
     macro avg       1.00      1.00      1.00       450
  weighted avg       1.00      1.00      1.00       450




                precision    recall  f1-score   support

         Fresh       1.00      1.00      1.00       150
Formalin-Mixed       1.00      1.00      1.00       150
        Rotten       1.00      1.00      1.00       150

      accuracy                           1.00       450
     macro avg       1.00      1.00      1.00       450
  weighted avg       1.00      1.00      1.00       450



## 3. XGBoost

In [4]:
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred, average='macro')
print(f"XGBoost  ->  Accuracy: {xgb_acc*100:.2f}%  |  Macro F1: {xgb_f1:.4f}")
print(classification_report(y_test, xgb_pred, target_names=TARGET_NAMES))

XGBoost  ->  Accuracy: 100.00%  |  Macro F1: 1.0000
                precision    recall  f1-score   support

         Fresh       1.00      1.00      1.00       150
Formalin-Mixed       1.00      1.00      1.00       150
        Rotten       1.00      1.00      1.00       150

      accuracy                           1.00       450
     macro avg       1.00      1.00      1.00       450
  weighted avg       1.00      1.00      1.00       450



## 4. LightGBM

In [5]:
lgb_model = lgb.LGBMClassifier(n_estimators=100, max_depth=4, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
lgb_acc = accuracy_score(y_test, lgb_pred)
lgb_f1 = f1_score(y_test, lgb_pred, average='macro')
print(f"LightGBM  ->  Accuracy: {lgb_acc*100:.2f}%  |  Macro F1: {lgb_f1:.4f}")
print(classification_report(y_test, lgb_pred, target_names=TARGET_NAMES))

LightGBM  ->  Accuracy: 100.00%  |  Macro F1: 1.0000
                precision    recall  f1-score   support

         Fresh       1.00      1.00      1.00       150
Formalin-Mixed       1.00      1.00      1.00       150
        Rotten       1.00      1.00      1.00       150

      accuracy                           1.00       450
     macro avg       1.00      1.00      1.00       450
  weighted avg       1.00      1.00      1.00       450



## 5. Sensor MLP (PyTorch)

In [6]:
class SensorMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 3)
        )
    def forward(self, x):
        return self.net(x)

# DataLoaders
train_ds = TensorDataset(torch.tensor(X_train_s, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val_s, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test_s, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

model = SensorMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Training
best_val_loss = float("inf")
best_weights = None
train_losses, val_losses = [], []

for epoch in range(1, 31):
    model.train()
    epoch_loss = 0
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(bx)
    train_losses.append(epoch_loss / len(train_ds))

    model.eval()
    vl = 0
    with torch.no_grad():
        for bx, by in val_loader:
            vl += criterion(model(bx), by).item() * len(bx)
    val_losses.append(vl / len(val_ds))

    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d}  |  Train Loss: {train_losses[-1]:.4f}  |  Val Loss: {val_losses[-1]:.4f}")

model.load_state_dict(best_weights)
print("Loaded best weights.")

Epoch 10  |  Train Loss: 0.0068  |  Val Loss: 0.0014
Epoch 20  |  Train Loss: 0.0018  |  Val Loss: 0.0002
Epoch 30  |  Train Loss: 0.0014  |  Val Loss: 0.0001
Loaded best weights.


Epoch 20  |  Train Loss: 0.0033  |  Val Loss: 0.0003


Epoch 30  |  Train Loss: 0.0016  |  Val Loss: 0.0001
Loaded best weights.


In [7]:
# Evaluate MLP on test set
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for bx, by in test_loader:
        all_preds.extend(torch.argmax(model(bx), dim=1).numpy())
        all_labels.extend(by.numpy())

mlp_acc = accuracy_score(all_labels, all_preds)
mlp_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Sensor MLP  ->  Accuracy: {mlp_acc*100:.2f}%  |  Macro F1: {mlp_f1:.4f}")
print(classification_report(all_labels, all_preds, target_names=TARGET_NAMES))

Sensor MLP  ->  Accuracy: 100.00%  |  Macro F1: 1.0000
                precision    recall  f1-score   support

         Fresh       1.00      1.00      1.00       150
Formalin-Mixed       1.00      1.00      1.00       150
        Rotten       1.00      1.00      1.00       150

      accuracy                           1.00       450
     macro avg       1.00      1.00      1.00       450
  weighted avg       1.00      1.00      1.00       450



## 6. Summary Comparison

In [8]:
results = pd.DataFrame([
    {'Model': 'Random Forest', 'Accuracy': rf_acc, 'Macro_F1': rf_f1},
    {'Model': 'XGBoost', 'Accuracy': xgb_acc, 'Macro_F1': xgb_f1},
    {'Model': 'LightGBM', 'Accuracy': lgb_acc, 'Macro_F1': lgb_f1},
    {'Model': 'Sensor MLP', 'Accuracy': mlp_acc, 'Macro_F1': mlp_f1},
])
results['Accuracy (%)'] = (results['Accuracy'] * 100).round(2)
results['Macro F1'] = results['Macro_F1'].round(4)
print(results[['Model', 'Accuracy (%)', 'Macro F1']].to_string(index=False))
results.to_csv(os.path.join(RESULTS_DIR, 'sensor_baselines.csv'), index=False)
print(f"\nSaved to {RESULTS_DIR}/sensor_baselines.csv")

        Model  Accuracy (%)  Macro F1
Random Forest         100.0       1.0
      XGBoost         100.0       1.0
     LightGBM         100.0       1.0
   Sensor MLP         100.0       1.0

Saved to ../results/sensor_baselines.csv


---
**Sensor baselines complete.** Proceed to `03_vision_baseline.ipynb`.